# 06 · Mapa de Whitechapel — tipografía y papel

**Objetivo:** corregir el solapamiento de nombres del `room_map.jpg` original y crear un mapa narrativo limpio para inspección, conservando **anclas de interacción (700,280,r175)** y las marcas del diseño actual. No utiliza modelos generativos: mapas, letras y pistas dibujadas por IA difusiva suelen deformarse.

El mapa nuevo es una **reconstrucción artística del mapa del videojuego**, **no un plano histórico cartográficamente certificado**. Verificar todos los nombres y posiciones con la documentación del juego antes de aprobar. Las cruces y el círculo están ubicados de forma aproximada según la ilustración existente. Comparar a resolución nativa 1280×720 y dentro del farol del juego. No cambiar las frases de las pistas ni el código de inspección.

Solo CPU; no consume GPU de Colab.

In [ ]:
%pip -q install 'Pillow>=10' 'numpy>=1.24' 'requests>=2.31'

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance, ImageOps, ImageChops
from IPython.display import display
import base64, io, json, math, hashlib, random, requests, shutil
REPO = "gabsvm/HorrorPlay"
COMMIT = "0681cfee36921ec2bd372f8ae8080a401f8bd63b"

def source(path):
    url = f"https://api.github.com/repos/{REPO}/contents/{path}"
    r = requests.get(url, params={"ref": COMMIT}, headers={"Accept":"application/vnd.github+json"}, timeout=90)
    r.raise_for_status()
    data = r.json()
    if data.get("encoding") != "base64":
        raise RuntimeError(f"GitHub no devolvió el JPG/PNG en base64: {path}")
    b = base64.b64decode(data["content"])
    result = Image.open(io.BytesIO(b)).convert("RGB" if path.lower().endswith((".jpg", ".jpeg")) else "RGBA")
    return result, hashlib.sha256(b).hexdigest()

def font(size=20, bold=False):
    files = ["/usr/share/fonts/truetype/dejavu/DejaVuSerif-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSerif.ttf",
             "/usr/share/fonts/truetype/liberation2/LiberationSerif-Regular.ttf"]
    for f in files:
        if Path(f).exists(): return ImageFont.truetype(f, size)
    return ImageFont.load_default()

def compare(images, thumb=(640,360), labels=None):
    w,h=thumb
    sheet=Image.new('RGB',(w*len(images),h+36),(17,18,20))
    dr=ImageDraw.Draw(sheet)
    for i,im in enumerate(images):
        sheet.paste(im.convert('RGB').resize((w,h), Image.Resampling.LANCZOS),(i*w,36))
        dr.text((i*w+12,10), labels[i] if labels else str(i),fill=(245,241,229),font=font(15))
    return sheet

def finish(folder, manifest, outputs):
    manifest.update({"repo":REPO,"source_commit":COMMIT,"status":"CANDIDATO - No integrado ni testeado en Android"})
    (folder/'manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
    archive=shutil.make_archive(str(folder),'zip',folder)
    print('Exportado:', archive, 'archivos:', len(list(folder.iterdir())))
    from google.colab import files
    files.download(archive)

In [ ]:
out=Path('/content/horrorplay_map_candidates');out.mkdir(parents=True,exist_ok=True)
old,sha=source('web/assets/closeups/room_map.jpg');assert old.size==(1280,720)
print('Original sha256:',sha)
display(old.resize((960,540)))

In [ ]:
from PIL import Image, ImageDraw, ImageFilter
import numpy as np
W,H=1280,720

def draw_map(seed=0, contrast=1):
    rng=np.random.default_rng(seed)
    yy,xx=np.mgrid[:H,:W]
    paper=np.stack([214-0.014*np.abs(xx-640),192-0.016*np.abs(xx-640),151-0.022*np.abs(xx-640)],axis=-1)
    paper+=rng.normal(0,3.5,(H,W,1))
    im=Image.fromarray(np.uint8(np.clip(paper,0,255)),'RGB')
    d=ImageDraw.Draw(im,'RGBA')
    ink=(49,40,33,235); faint=(94,73,46,65)
    # Bordes del plano, cuadrícula de parcelas puramente decorativa.
    d.rectangle((32,32,1247,685),outline=(84,65,40,105),width=2)
    for i in range(38):
        x=50+(i*137)%1150;y=110+(i*73)%460
        d.rectangle((x,y,x+33+(i%4)*11,y+22+(i%3)*9),outline=faint,width=1)
    # Ejes heredados de la lectura visual del original; NO son cartografía topológica exacta.
    routes=[[(72,375),(1197,375)],[(490,75),(490,592)],[(785,76),(785,591)],
            [(944,389),(944,590)],[(335,212),(1000,212)],[(475,257),(850,257)],
            [(489,374),(1200,577)]]
    for pts in routes:
        d.line(pts,fill=(241,221,179,195),width=14,joint='curve')
        d.line(pts,fill=ink,width=4,joint='curve')
    # Títulos colocados en bandas vacías para evitar solapamientos.
    def write(x,y,t,size=22,bold=False,fill=(62,49,39,245)):
        d.text((x,y),t,font=font(max(size,22),bold),fill=fill)
    write(868,47,'WHITECHAPEL',32,True)
    write(880,91,'EAST END · 1888',22)
    write(100,70,'SPITALFIELDS',24,True)
    write(515,144,'CHRIST CHURCH',19,True)
    write(510,180,'HANBURY ST',20)
    write(750,231,'FLOWER & DEAN ST',17)
    write(87,338,'MITRE SQUARE',18,True)
    write(91,392,'WHITECHAPEL HIGH ST',19)
    write(885,336,'WHITECHAPEL RD',18)
    write(960,468,'BERNER ST',18)
    write(650,497,'COMMERCIAL RD',21)
    write(881,115,"BUCK'S ROW",18)
    write(589,312,'GOULSTON ST',15)
    write(549,579,'LONDON HOSPITAL',18)
    # Marcadores narrativos (los mismos centros aproximados del JPG original).
    markers=[('X',145,430),('X',640,202),('X',735,258),('X',895,145),('X',950,488)]
    for symbol,x,y in markers:
        r=15;d.line((x-r,y-r,x+r,y+r),fill=(60,57,54,230),width=10)
        d.line((x-r,y+r,x+r,y-r),fill=(60,57,54,230),width=10)
    d.ellipse((622,356,658,394),outline=(65,61,57,220),width=9)
    d.line((640,355,640,393),fill=(65,61,57,180),width=2)
    # Rosa de los vientos y escala en zonas protegidas lejos del hotspot central.
    d.polygon([(76,592),(65,623),(76,644),(87,623)],fill=(61,52,45,225))
    write(70,654,'N',18,True)
    d.line((1006,644,1162,644),fill=ink,width=3)
    for x in (1006,1045,1084,1123,1162):d.line((x,635,x,653),fill=ink,width=2)
    write(1022,658,'QUARTER OF A MILE',14)
    if contrast>1:im=ImageEnhance.Contrast(im).enhance(contrast)
    # Borde desgastado y granulado, reproducibles por seed.
    im=ImageEnhance.Color(im).enhance(.86)
    return im

versions=[draw_map(1888,1.0),draw_map(1911,1.07)]
for name,im in zip(['A_claro','B_tinta'],versions):
    im.save(out/f'room_map_{name}.png')
    im.save(out/f'room_map_{name}.jpg',quality=90,optimize=True)
sheet=compare([old,*versions],(640,360),['Actual','A: legible','B: más tinta'])
sheet.save(out/'AB_map_640.png');display(sheet)

## Control del hotspot

El punto inspeccionable sigue siendo el círculo `(700,280,r175)` de `room.js`. Su posición no se cambia. Se dibuja como guía **sobre una copia de revisión**, no sobre el JPG final. Verifica que la cartela y rosa de los vientos no formen parte del círculo. Revisa también la posición y ortografía histórica de los rótulos antes de aceptar.

In [ ]:
review=versions[0].copy().convert('RGBA')
layer=Image.new('RGBA',review.size,(0,0,0,0))
rd=ImageDraw.Draw(layer)
rd.ellipse((700-175,280-175,700+175,280+175),outline=(200,15,18,240),width=5)
review=Image.alpha_composite(review,layer)
display(review.resize((960,540)))
review.save(out/'QA_hotspot_MAP_NO_INTEGRAR.png')
finish(out,{'source_sha256':sha,'hotspot_center':[700,280],'hotspot_radius':175,
  'warning':'Mapa artistico no verificado geograficamente: revisar nombres, lugares, pistas'},[])